# Streaming

<img src="./assets/LC_streaming.png" width="400">

Streaming reduces the latency between generating data and the user receiving it.
There are two types frequently used with Agents:

https://docs.langchain.com/oss/python/langchain/streaming/overview

## Setup

Load and/or check for needed environmental variables

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")

Checking DEEPSEEK_API_KEY... DEEPSEEK_API_KEY=****bdb0
Checking LANGSMITH_API_KEY... LANGSMITH_API_KEY=****0714
Checking LANGSMITH_TRACING... LANGSMITH_TRACING=true
Checking LANGSMITH_PROJECT... LANGSMITH_PROJECT=****ials


In [2]:
from langchain.agents import create_agent

In [3]:
agent = create_agent(
    model="deepseek:deepseek-chat",
    system_prompt="You are a full-stack comedian",
)

## No Streaming (invoke)

In [6]:
result = agent.invoke({"messages": [{"role": "user", "content": "Tell me a joke"}]})
print(result["messages"][1].content)

Why don't scientists trust atoms?

Because they make up everything!


## values
You have seen this streaming mode in our examples so far. 

In [7]:
# Stream = values
for step in agent.stream(
    {"messages": [{"role": "user", "content": "Tell me a Dad joke"}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Tell me a Dad joke
================================== Ai Message ==================================

Why don't eggs tell jokes?  
Because they'd crack each other up.


## messages
Messages stream data token by token - the lowest latency possible. This is perfect for interactive applications like chatbots.

In [8]:
for token, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "Write me a family friendly poem."}]},
    stream_mode="messages",
):
    print(f"{token.content}", end="")

In a house with a roof of slate,
Lived a family of five, who woke at eight.
First came Dad, with a morning cheer,
Whistling a tune for all to hear.
Mom followed after, with a yawn and a smile,
Brewing a pot of coffee, mile by mile.

Then pitter-patter, a thunderous sound,
Three pairs of feet came hitting the ground.
There was Lily, with her hair in a whirl,
And Ben, who gave his toy truck a twirl.
And little Sam, still clutching his bear,
Rubbing the sleep from his eyes with care.

They gathered round the table, a cozy bunch,
For pancakes piled in a syrupy lunch.
There was laughter and a splash of milk,
And a joke that made the whole table tilt.
They talked of dreams and plans for the day,
In their own wonderful, muddled way.

Later, with bikes and a bright red ball,
They played together, having a ball.
They built a fort with a sheet and a chair,
A castle for a knight and a great brown bear.
They shared their snacks and they shared their games,
And no two moments were quite the same.



## Tools can stream too!
Streaming generally means delivering information to the user before the final result is ready. There are many cases where this is useful. A `get_stream_writer` writer allows you to easily stream `custom` data from sources you create.

In [9]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()
    # stream any arbitrary data
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"It's always sunny in {city}!"


agent = create_agent(
    model="deepseek:deepseek-chat",
    tools=[get_weather],
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["values", "custom"],
):
    print(chunk)

('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='748a1319-ee91-47b1-92bd-6818d6ecbf07')]})
('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='748a1319-ee91-47b1-92bd-6818d6ecbf07'), AIMessage(content='I can help you check the weather in SF (San Francisco). Let me get that information for you.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 305, 'total_tokens': 370, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 305}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache', 'id': 'a8462f83-2413-41fe-9116-a8e45ed9ec0b', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019bdaed-fa73-7852-

In [8]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["custom"],
):
    print(chunk)

('custom', 'Looking up data for city: San Francisco')
('custom', 'Acquired data for city: San Francisco')


## Try different modes on your own!
Modify the stream mode and the select to produce different results.

In [ ]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["values", "custom"],
):
    if chunk[0] == "custom":
        print(chunk[1])